***Подготовка***

In [6]:
!pip install -q dash

In [7]:
import gzip
import re
from collections import Counter
from datetime import timedelta

import pandas as pd
import plotly.express as px

from dash import Dash, html, dcc, Input, Output
from google.colab import output as colab_output

## А0. Грузим и фильтруем данные

In [8]:
REVIEWS_PATH = 'reviews_Toys_and_Games_5.json.gz'
META_PATH = 'meta_Toys_and_Games.json.gz'

PIECES_RE = re.compile(r'(\d+)[\s-]*pieces?', re.IGNORECASE)

def has_puzzles(cats):
    # рекурсивно ищем Puzzles на любом уровне
    if isinstance(cats, str):
        return cats == 'Puzzles'
    if isinstance(cats, list):
        return any(has_puzzles(x) for x in cats)
    return False

def category_leaf(cats):
    # самый глубокий элемент
    if not cats:
        return None
    if isinstance(cats, list) and cats:
        last = cats[-1]
        if isinstance(last, list) and last:
            return last[-1]
        return last
    return cats

def to_float(x):
    try:
        return float(x)
    except:
        return float('nan')

def extract_pieces(title):
    if not title:
        return None
    m = PIECES_RE.search(title)
    return int(m.group(1)) if m else None

In [9]:
# мета: 336к товаров, сразу фильтруем пазлы
meta_rows = []
with gzip.open(META_PATH, 'r') as g:
    for line in g:
        try:
            d = eval(line)
        except:
            continue
        if has_puzzles(d.get('categories', [])):
            meta_rows.append({
                'asin': d.get('asin'),
                'title': d.get('title'),
                'brand': d.get('brand'),
                'price': to_float(d.get('price')),
                'category_leaf': category_leaf(d.get('categories', [])),
            })

meta = pd.DataFrame(meta_rows)
print('товаров пазлов:', len(meta))
meta.head()

товаров пазлов: 14427


,asin,title,brand,price,category_leaf
0,0000191639,Dr. Suess 19163 Dr. Seuss Puzzle 3 Pack Bundle,Dr. Seuss,37.12,Jigsaw Puzzles
1,0375829695,Dr. Seuss Jigsaw Puzzle Book: With Six 48-Piec...,Dr. Seuss,24.82,Jigsaw Puzzles
2,0439400066,3D Puzzle Buster,None,99.15,3-D Puzzles
3,0439845297,Insect Model Lytta Stygica Green Flower Beetle...,None,8.99,3-D Puzzles
4,0545302285,Scholastic Teacher's Friend Rhyming Learning P...,Scholastic Teacher&#39;s Friend,8.94,Jigsaw Puzzles


In [10]:
# отзывы: оставляем только по нашим asin
asins = set(meta['asin'].dropna())
review_rows = []
with gzip.open(REVIEWS_PATH, 'r') as g:
    for line in g:
        try:
            d = eval(line)
        except:
            continue
        if d.get('asin') in asins:
            review_rows.append({
                'reviewerID': d.get('reviewerID'),
                'reviewerName': d.get('reviewerName'),
                'asin': d.get('asin'),
                'overall': d.get('overall'),
                'summary': d.get('summary'),
                'unixReviewTime': d.get('unixReviewTime'),
            })

reviews = pd.DataFrame(review_rows)
print('отзывов на пазлы:', len(reviews))
reviews.head()

отзывов на пазлы: 6998


,reviewerID,reviewerName,asin,overall,summary,unixReviewTime
0,ABKGN0ETNUNOA,Beth S,073533305X,5.0,Adorable puzzle,1391731200
1,A3FFY8SLBK89ES,"B. Rachal ""msbrachal""",073533305X,5.0,great puzzle,1390262400
2,A2CDP7ULSHS5G9,BuyerG,073533305X,5.0,Developmentally Appropriate - Develops great s...,1388188800
3,ASSHPKQBFZPB5,"M. C. ""lovestoread""",073533305X,5.0,Great,1384473600
4,A3U6SWQW5ID7F5,PNaz,073533305X,5.0,We love this puzzle!,1390867200


In [11]:
# джойн
df = reviews.merge(meta, on='asin', how='inner')
df['review_date'] = pd.to_datetime(df['unixReviewTime'], unit='s')
df['pieces'] = df['title'].apply(extract_pieces)
print('строк после джойна:', len(df))
df.head()

строк после джойна: 6998


,reviewerID,reviewerName,asin,overall,summary,unixReviewTime,title,brand,price,category_leaf,review_date,pieces
0,ABKGN0ETNUNOA,Beth S,073533305X,5.0,Adorable puzzle,1391731200,Construction Jumbo Puzzle,Mudpuppy,9.56,Floor Puzzles,2014-02-07,NaN
1,A3FFY8SLBK89ES,"B. Rachal ""msbrachal""",073533305X,5.0,great puzzle,1390262400,Construction Jumbo Puzzle,Mudpuppy,9.56,Floor Puzzles,2014-01-21,NaN
2,A2CDP7ULSHS5G9,BuyerG,073533305X,5.0,Developmentally Appropriate - Develops great s...,1388188800,Construction Jumbo Puzzle,Mudpuppy,9.56,Floor Puzzles,2013-12-28,NaN
3,ASSHPKQBFZPB5,"M. C. ""lovestoread""",073533305X,5.0,Great,1384473600,Construction Jumbo Puzzle,Mudpuppy,9.56,Floor Puzzles,2013-11-15,NaN
4,A3U6SWQW5ID7F5,PNaz,073533305X,5.0,We love this puzzle!,1390867200,Construction Jumbo Puzzle,Mudpuppy,9.56,Floor Puzzles,2014-01-28,NaN


In [12]:
# таблица юзеров (уникальные reviewerID)
users = df.groupby('reviewerID').agg(
    reviewer_name=('reviewerName', 'first'),
    first_review_date=('review_date', 'min'),
    n_reviews=('reviewerID', 'size'),
).reset_index()
print('юзеров:', len(users))
users.head()

юзеров: 3901


,reviewerID,reviewer_name,first_review_date,n_reviews
0,A09027871HUIYHTFHPJX6,Leslie Hawkins,2013-03-11,1
1,A101C99CG8EFUH,Benjamin McGough,2011-03-28,1
2,A102D49JO9BOZY,Julie O.,2013-01-15,1
3,A1051DBTLWP5A2,Monkey Momma,2012-01-11,1
4,A105LFG3MT4N23,TM,2011-06-26,1


## А1. Класс UserProfile

In [13]:
class UserProfile:

    # точка отсчета "сейчас" для visit_frequency. По умолчанию реальное сейчас,
    # но датасет 2001-2014 - подменяем на конец датасета в демо-ячейке.
    REF_DATE = pd.Timestamp.now()

    def __init__(self, reviewer_id, name, registration_date, n_reviews):
        self.reviewer_id = reviewer_id
        self.name = None if pd.isna(name) else name
        self.registration_date = pd.to_datetime(registration_date)
        self.n_reviews = int(n_reviews)

    @property
    def visit_frequency(self):
        # отзывов в день с момента первого отзыва до даты-среза
        days = (self.REF_DATE - self.registration_date).days
        if days <= 0:
            return 0.0
        return self.n_reviews / days

    @classmethod
    def from_row(cls, row):
        return cls(
            reviewer_id=row['reviewerID'],
            name=row['reviewer_name'],
            registration_date=row['first_review_date'],
            n_reviews=row['n_reviews'],
        )

    def __repr__(self):
        return f'UserProfile(id={self.reviewer_id!r}, name={self.name!r}, n_reviews={self.n_reviews})'

    def __eq__(self, other):
        if not isinstance(other, UserProfile):
            return NotImplemented
        return self.reviewer_id == other.reviewer_id

    def __hash__(self):
        return hash(self.reviewer_id)

In [14]:
# подменяем REF_DATE на конец датасета чтоб visit_frequency был осмысленный
# (датасет 2001-2014, реальное сейчас даст 10+ лет пустоты в знаменателе)
UserProfile.REF_DATE = df['review_date'].max() + timedelta(days=60)
print('REF_DATE для UserProfile:', UserProfile.REF_DATE.date())
print()

# 5 профилей из первых строк
for _, row in users.head(5).iterrows():
    u = UserProfile.from_row(row)
    print(u)
    print(f'  registration={u.registration_date.date()} | visit_frequency={round(u.visit_frequency, 5)}')

# проверка __eq__ и __hash__
u1 = UserProfile.from_row(users.iloc[0])
u2 = UserProfile.from_row(users.iloc[0])
print()
print('равны:', u1 == u2, '| hash одинаковый:', hash(u1) == hash(u2))

REF_DATE для UserProfile: 2014-09-17

UserProfile(id='A09027871HUIYHTFHPJX6', name='Leslie Hawkins', n_reviews=1)
  registration=2013-03-11 | visit_frequency=0.0018
UserProfile(id='A101C99CG8EFUH', name='Benjamin McGough', n_reviews=1)
  registration=2011-03-28 | visit_frequency=0.00079
UserProfile(id='A102D49JO9BOZY', name='Julie O.', n_reviews=1)
  registration=2013-01-15 | visit_frequency=0.00164
UserProfile(id='A1051DBTLWP5A2', name='Monkey Momma', n_reviews=1)
  registration=2012-01-11 | visit_frequency=0.00102
UserProfile(id='A105LFG3MT4N23', name='TM', n_reviews=1)
  registration=2011-06-26 | visit_frequency=0.00085

равны: True | hash одинаковый: True


## А2. Класс Order


In [15]:
class Order:

    def __init__(self, reviewer_id, asin, date, price, title=None, brand=None,
                 category=None, pieces=None, rating=None, summary=None):
        self.reviewer_id = reviewer_id
        self.asin = asin
        self.date = pd.to_datetime(date)
        self.price = float(price) if not pd.isna(price) else None
        self.title = None if pd.isna(title) else title
        self.brand = None if pd.isna(brand) else brand
        self.category = None if pd.isna(category) else category
        self.pieces = int(pieces) if not pd.isna(pieces) else None
        self.rating = float(rating) if not pd.isna(rating) else None
        self.summary = None if pd.isna(summary) else summary

    @property
    def total(self):
        # без скидок и количества - просто цена
        return self.price

    def user(self, registry=None):
        # если передали словарь {id: UserProfile} - вернем профиль, иначе id
        if registry is not None:
            return registry.get(self.reviewer_id)
        return self.reviewer_id

    @classmethod
    def from_row(cls, row):
        return cls(
            reviewer_id=row['reviewerID'],
            asin=row['asin'],
            date=row['review_date'],
            price=row.get('price'),
            title=row.get('title'),
            brand=row.get('brand'),
            category=row.get('category_leaf'),
            pieces=row.get('pieces'),
            rating=row.get('overall'),
            summary=row.get('summary'),
        )

    def __repr__(self):
        d = self.date.date() if pd.notna(self.date) else None
        return f'Order(user={self.reviewer_id!r}, asin={self.asin!r}, date={d}, total={self.total})'

    def __eq__(self, other):
        if not isinstance(other, Order):
            return NotImplemented
        return (self.reviewer_id == other.reviewer_id
                and self.asin == other.asin
                and self.date == other.date)

    def __hash__(self):
        return hash((self.reviewer_id, self.asin, self.date))

In [16]:
# 5 заказов
for _, row in df.head(5).iterrows():
    o = Order.from_row(row)
    print(o)
    print(f'  category={o.category} | brand={o.brand} | rating={o.rating} | pieces={o.pieces}')


print()
registry = {row['reviewerID']: UserProfile.from_row(row) for _, row in users.head(100).iterrows()}
sample_row = df[df['reviewerID'].isin(registry)].iloc[0]
o = Order.from_row(sample_row)
print('o.user(registry):', o.user(registry))
print('o.user() без registry:', o.user())

Order(user='ABKGN0ETNUNOA', asin='073533305X', date=2014-02-07, total=9.56)
  category=Floor Puzzles | brand=Mudpuppy | rating=5.0 | pieces=None
Order(user='A3FFY8SLBK89ES', asin='073533305X', date=2014-01-21, total=9.56)
  category=Floor Puzzles | brand=Mudpuppy | rating=5.0 | pieces=None
Order(user='A2CDP7ULSHS5G9', asin='073533305X', date=2013-12-28, total=9.56)
  category=Floor Puzzles | brand=Mudpuppy | rating=5.0 | pieces=None
Order(user='ASSHPKQBFZPB5', asin='073533305X', date=2013-11-15, total=9.56)
  category=Floor Puzzles | brand=Mudpuppy | rating=5.0 | pieces=None
Order(user='A3U6SWQW5ID7F5', asin='073533305X', date=2014-01-28, total=9.56)
  category=Floor Puzzles | brand=Mudpuppy | rating=5.0 | pieces=None

o.user(registry): UserProfile(id='A10E3F50DIUJEE', name='C Wahlman "cdub"', n_reviews=5)
o.user() без registry: A10E3F50DIUJEE


## А3. Класс UserAnalytics

Метрики по одному юзеру. Принимает UserProfile + список Order этого юзера.

Считаем:
- **LTV** - сумма total по всем заказам
- **AOV** - средний чек
- **churn** - True если последний заказ > 90 дней назад
- **retention_30/60/90** - вернулся ли в окно X дней после первого заказа
- **rfm_segment** - Champion / Loyal / Promising / At Risk / Lost (по скорам R, F, M по 1-5)
- **favorite_category** - чаще всего покупаемая категория
- **complexity_preference** - средний размер пазла + ярлык (легкие/средние/сложные)


In [17]:
class UserAnalytics:

    # точка отсчета для recency. По умолчанию реальное сейчас.
    REF_DATE = pd.Timestamp.now()

    def __init__(self, user, orders):
        self._user = user
        self._orders = list(orders)

    @property
    def user(self):
        return self._user

    @property
    def orders(self):
        return self._orders

    @property
    def ltv(self):
        return sum(o.total for o in self._orders if o.total is not None)

    @property
    def aov(self):
        if not self._orders:
            return 0.0
        return self.ltv / len(self._orders)

    @property
    def first_order_date(self):
        dates = [o.date for o in self._orders if pd.notna(o.date)]
        return min(dates) if dates else None

    @property
    def last_order_date(self):
        dates = [o.date for o in self._orders if pd.notna(o.date)]
        return max(dates) if dates else None

    @property
    def churn(self):
        last = self.last_order_date
        if last is None:
            return True
        return (self.REF_DATE - last).days > 90

    def _retention_window(self, days):
        # вернулся ли в окно X дней после первого заказа
        first = self.first_order_date
        if first is None:
            return False
        threshold = first + timedelta(days=days)
        return any(first < o.date <= threshold for o in self._orders if pd.notna(o.date))

    @property
    def retention_30(self):
        return self._retention_window(30)

    @property
    def retention_60(self):
        return self._retention_window(60)

    @property
    def retention_90(self):
        return self._retention_window(90)

    def _rfm_scores(self):
        # R - давность последнего заказа
        last = self.last_order_date
        if last is None:
            r = 1
        else:
            days = (self.REF_DATE - last).days
            if days < 30:
                r = 5
            elif days < 90:
                r = 4
            elif days < 180:
                r = 3
            elif days < 365:
                r = 2
            else:
                r = 1

        # F - число заказов
        n = len(self._orders)
        if n >= 10:
            f = 5
        elif n >= 5:
            f = 4
        elif n >= 3:
            f = 3
        elif n == 2:
            f = 2
        else:
            f = 1

        # M - выручка
        ltv = self.ltv
        if ltv >= 500:
            m = 5
        elif ltv >= 200:
            m = 4
        elif ltv >= 100:
            m = 3
        elif ltv >= 50:
            m = 2
        else:
            m = 1

        return r, f, m

    @property
    def rfm_segment(self):
        r, f, m = self._rfm_scores()
        if r >= 4 and f >= 4 and m >= 4:
            return 'Champion'
        if r >= 3 and f >= 3:
            return 'Loyal'
        if r >= 4 and f <= 2:
            return 'Promising'
        if r <= 2 and f >= 3:
            return 'At Risk'
        return 'Lost'

    @property
    def favorite_category(self):
        cats = [o.category for o in self._orders if o.category is not None]
        if not cats:
            return None
        return Counter(cats).most_common(1)[0][0]

    @property
    def complexity_preference(self):
        pcs = [o.pieces for o in self._orders if o.pieces is not None]
        if not pcs:
            return None
        avg = sum(pcs) / len(pcs)
        if avg < 100:
            level = 'легкие'
        elif avg < 500:
            level = 'средние'
        else:
            level = 'сложные'
        return {'avg_pieces': round(avg, 1), 'level': level}

    def __len__(self):
        return len(self._orders)

    def __iter__(self):
        return iter(self._orders)

    def __repr__(self):
        return (f'UserAnalytics(user={self._user.reviewer_id!r}, '
                f'orders={len(self._orders)}, ltv={self.ltv:.2f}, '
                f'segment={self.rfm_segment!r})')

In [18]:

UserAnalytics.REF_DATE = df['review_date'].max() + timedelta(days=60)
print('REF_DATE:', UserAnalytics.REF_DATE.date())

# берем самого активного юзера
counts = df.groupby('reviewerID').size().sort_values(ascending=False)
top_id = counts.index[0]
print('самый активный:', top_id, '| заказов:', counts.iloc[0])

user = UserProfile.from_row(users[users['reviewerID'] == top_id].iloc[0])
user_orders = [Order.from_row(r) for _, r in df[df['reviewerID'] == top_id].iterrows()]
a = UserAnalytics(user, user_orders)
a

REF_DATE: 2014-09-17
самый активный: A1EVV74UQYVKRY | заказов: 22


UserAnalytics(user='A1EVV74UQYVKRY', orders=22, ltv=402.56, segment='Champion')

In [19]:
# все метрики словарем
{
    'ltv': round(a.ltv, 2),
    'aov': round(a.aov, 2),
    'first_order': a.first_order_date.date(),
    'last_order': a.last_order_date.date(),
    'churn': a.churn,
    'retention_30/60/90': (a.retention_30, a.retention_60, a.retention_90),
    'rfm_scores (R, F, M)': a._rfm_scores(),
    'segment': a.rfm_segment,
    'favorite_category': a.favorite_category,
    'complexity': a.complexity_preference,
    'len(a)': len(a),
}

{'ltv': 402.56,
 'aov': 18.3,
 'first_order': datetime.date(2009, 1, 11),
 'last_order': datetime.date(2014, 7, 8),
 'churn': False,
 'retention_30/60/90': (False, False, False),
 'rfm_scores (R, F, M)': (4, 5, 4),
 'segment': 'Champion',
 'favorite_category': 'Jigsaw Puzzles',
 'complexity': {'avg_pieces': 928.0, 'level': 'сложные'},
 'len(a)': 22}

## А4. Класс BusinessDashboard + дашборд на dash


In [20]:
class BusinessDashboard:

    def __init__(self, analytics):
        self._analytics = list(analytics)

    @property
    def analytics(self):
        return self._analytics

    @property
    def total_revenue(self):
        return sum(a.ltv for a in self._analytics)

    @property
    def avg_ltv(self):
        if not self._analytics:
            return 0.0
        return self.total_revenue / len(self._analytics)

    @property
    def churn_rate(self):
        if not self._analytics:
            return 0.0
        return sum(1 for a in self._analytics if a.churn) / len(self._analytics)

    @property
    def rfm_breakdown(self):
        return dict(Counter(a.rfm_segment for a in self._analytics))

    def top_categories(self, n=10):
        cats = []
        for a in self._analytics:
            for o in a:
                if o.category is not None:
                    cats.append(o.category)
        return Counter(cats).most_common(n)

    def top_brands(self, n=10):
        brands = []
        for a in self._analytics:
            for o in a:
                if o.brand is not None:
                    brands.append(o.brand)
        return Counter(brands).most_common(n)

    @property
    def revenue_by_year(self):
        rev = {}
        for a in self._analytics:
            for o in a:
                if o.total is not None and pd.notna(o.date):
                    y = o.date.year
                    rev[y] = rev.get(y, 0) + o.total
        return dict(sorted(rev.items()))

    @property
    def revenue_by_category(self):
        rev = {}
        for a in self._analytics:
            for o in a:
                if o.total is not None and o.category is not None:
                    rev[o.category] = rev.get(o.category, 0) + o.total
        return rev

    def __len__(self):
        return len(self._analytics)

    def __repr__(self):
        return (f'BusinessDashboard(users={len(self._analytics)}, '
                f'revenue={self.total_revenue:.2f})')

In [21]:
# собираем профили + аналитику по всем юзерам
profiles_all = {row['reviewerID']: UserProfile.from_row(row) for _, row in users.iterrows()}

analytics_list = []
for rid, grp in df.groupby('reviewerID'):
    user_orders = [Order.from_row(r) for _, r in grp.iterrows()]
    analytics_list.append(UserAnalytics(profiles_all[rid], user_orders))

bd = BusinessDashboard(analytics_list)
bd

BusinessDashboard(users=3901, revenue=106374.10)

In [22]:
# базовые цифры
print(f'total revenue: ${bd.total_revenue:,.2f}')
print(f'avg LTV: ${bd.avg_ltv:,.2f}')
print(f'churn rate: {bd.churn_rate * 100:.1f}%')
print(f'юзеров: {len(bd):,}')
print()
print('RFM:', bd.rfm_breakdown)
print()
print('топ-10 категорий:')
for c, n in bd.top_categories(10):
    print(f'  {c:35s} -> {n}')
print()
print('топ-5 брендов:')
for b, n in bd.top_brands(5):
    print(f'  {b:35s} -> {n}')

total revenue: $106,374.10
avg LTV: $27.27
churn rate: 98.0%
юзеров: 3,901

RFM: {'Lost': 3135, 'At Risk': 658, 'Loyal': 44, 'Promising': 63, 'Champion': 1}

топ-10 категорий:
  Jigsaw Puzzles                      -> 3313
  Pegged Puzzles                      -> 1149
  Floor Puzzles                       -> 963
  Maze & Sequential Puzzles           -> 438
  3-D Puzzles                         -> 350
  Puzzles                             -> 299
  Puzzle Accessories                  -> 199
  Assembly & Disentanglement Puzzles  -> 172
  Puzzle Boxes                        -> 65
  Brain Teasers                       -> 40

топ-5 брендов:
  Melissa &amp; Doug                  -> 2610
  Ravensburger                        -> 2349
  PlaSmart                            -> 211
  Pastime Puzzles                     -> 174
  Think Fun                           -> 161


In [23]:
r = bd.rfm_breakdown
px.bar(x=list(r.keys()), y=list(r.values()),
       title='RFM сегменты', labels={'x': 'сегмент', 'y': 'юзеров'})

In [24]:
cats = bd.top_categories(10)
px.bar(x=[c[0] for c in cats], y=[c[1] for c in cats],
       title='Топ-10 категорий по числу заказов',
       labels={'x': 'категория', 'y': 'заказов'})

In [25]:
y = bd.revenue_by_year
px.bar(x=list(y.keys()), y=list(y.values()),
       title='Выручка по годам', labels={'x': 'год', 'y': 'выручка $'})

### Интерактивный дашборд на dash

In [26]:

# готовим фигуры
_r = bd.rfm_breakdown
fig_rfm = px.bar(x=list(_r.keys()), y=list(_r.values()),
                 title='RFM сегменты', labels={'x': 'сегмент', 'y': 'юзеров'})

_c = bd.top_categories(10)
fig_cat = px.bar(x=[c[0] for c in _c], y=[c[1] for c in _c],
                 title='Топ категорий', labels={'x': 'категория', 'y': 'заказов'})


def card(title, value):
    return html.Div([
        html.Div(title, style={'fontSize': 13, 'color': '#666'}),
        html.Div(value, style={'fontSize': 24, 'fontWeight': 'bold', 'marginTop': '4px'}),
    ], style={'padding': '15px', 'borderRadius': '8px', 'background': '#f5f5f5',
              'flex': 1, 'margin': '5px', 'textAlign': 'center'})


app = Dash()
app.layout = html.Div([
    html.H2('Пазлы - дашборд', style={'textAlign': 'center'}),
    html.Div([
        card('Общая выручка', f'${bd.total_revenue:,.0f}'),
        card('Avg LTV', f'${bd.avg_ltv:,.1f}'),
        card('Churn rate', f'{bd.churn_rate * 100:.1f}%'),
        card('Юзеров', f'{len(bd):,}'),
    ], style={'display': 'flex'}),
    html.Div([
        dcc.Graph(figure=fig_rfm, style={'flex': 1}),
        dcc.Graph(figure=fig_cat, style={'flex': 1}),
    ], style={'display': 'flex'}),
    html.Div([
        html.Label('Срез выручки: ', style={'marginRight': '10px'}),
        dcc.Dropdown(
            id='cut-dd',
            options=[
                {'label': 'по годам', 'value': 'year'},
                {'label': 'по категориям', 'value': 'cat'},
            ],
            value='year',
            style={'width': '300px'},
        ),
    ], style={'display': 'flex', 'alignItems': 'center', 'padding': '10px'}),
    dcc.Graph(id='cut-graph'),
], style={'fontFamily': 'sans-serif', 'maxWidth': '1200px', 'margin': '0 auto'})


@app.callback(Output('cut-graph', 'figure'), Input('cut-dd', 'value'))
def update_cut(val):
    if val == 'year':
        r = bd.revenue_by_year
        return px.bar(x=list(r.keys()), y=list(r.values()),
                      title='Выручка по годам',
                      labels={'x': 'год', 'y': 'выручка $'})
    r = bd.revenue_by_category
    return px.bar(x=list(r.keys()), y=list(r.values()),
                  title='Выручка по категориям',
                  labels={'x': 'категория', 'y': 'выручка $'})


# запускаем сервер в фоне и пробрасываем порт через прокси Colab
import threading
threading.Thread(target=lambda: app.run(port=8050, debug=False), daemon=True).start()

# даём серверу пару секунд подняться, потом показываем iframe
import time
time.sleep(2)
colab_output.serve_kernel_port_as_iframe(8050, height='900px')

Dash is running on http://127.0.0.1:8050/



INFO:dash.dash:Dash is running on http://127.0.0.1:8050/



 * Serving Flask app '__main__'
 * Debug mode: off


INFO:werkzeug:WARNING: This is a development server. Do not use it in a production deployment. Use a production WSGI server instead.
 * Running on http://127.0.0.1:8050
INFO:werkzeug:Press CTRL+C to quit


<IPython.core.display.Javascript object>

### Что мы видим из дашборда

**Объём данных:**
- 14 427 уникальных пазлов в каталоге
- 6 998 отзывов-заказов от 3 901 пользователя
- 1.79 заказа на юзера в среднем, медиана 1, то есть большинство покупает раз и уходит

**Деньги:**
- Совокупная выручка по выборке $106 374
- Средний LTV $27.27
- Churn rate 98% (REF_DATE = 17.09.2014, окно 90 дней) - почти весь хвост датасета это "ушедшие"

**Концентрация рынка:**
- Топ-2 бренда (Melissa & Doug + Ravensburger) держат 71% всех заказов (4 959 из 6 998)
- Jigsaw Puzzles один забирает 47% (3 313 заказов), топ-3 категории (Jigsaw + Pegged + Floor) дают 78%
- Длинный хвост, т.к. 3-D Puzzles, Puzzle Boxes, Brain Teasers получают единичные заказы

**RFM-сегменты:**
- Lost: 3 135 юзеров (80%) - один заказ давно и тишина
- At Risk: 658 - раньше покупали активно, потом ушли
- Loyal + Promising: 107 - живое ядро
- Champion: 1 - аномалия (юзер A1EVV74UQYVKRY, 22 заказа за 5+ лет, LTV $402)

**Что это говорит про рынок:**

Данные старые (2014), но картина типичная для нишевого товарного рынка - высокая концентрация на 2-3 крупных бренда, доминирование "детского" сегмента (Pegged + Floor + большая доля Jigsaw для младших) и единичная активность массового покупателя. Бизнес-модель строится на acquisition нового юзера (один пазл купил - ушел), а не на retention. Загнать средний чек выше $15 или вернуть юзера за вторым пазлом сложно: пазл это разовая покупка-подарок или элемент досуга на месяц.

## А5. Парсинг Steam через Selenium


Сначала пробовал тег Puzzle (1664), но туда лезут AAA-игры с щепоткой загадок (Hogwarts Legacy, Stray, HITMAN). Card Game (тег 1666) дает строго карточные: Slay the Spire, Balatro, Magic, Inscryption, Tabletop Simulator, Marvel Snap и т.д.

**Что тащим со страницы поиска:** название, дата выхода, обзор-сводка (Very Positive / Mixed / etc.), процент позитивных, число отзывов, цена, app_id.

**Почему именно Selenium:** Steam грузит карточки через JS, requests+BS4 отдает пустой каркас. С Selenium ждем рендера и берем готовый DOM.

In [ ]:
# Ставим Chrome и Selenium. В Colab Chrome не предустановлен - тащим .deb с офсайта.
# Локально (Win/Mac) apt-get просто проигнорируется, останется только pip install.
import sys
IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    !apt-get update -qq
    !wget -q https://dl.google.com/linux/direct/google-chrome-stable_current_amd64.deb -O /tmp/chrome.deb
    !apt-get install -y -qq /tmp/chrome.deb

!pip install -q selenium

INFO:werkzeug:127.0.0.1 - - [26/May/2026 09:42:48] "GET / HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [26/May/2026 09:42:48] "GET /_dash-component-suites/dash/deps/react@18.v4_1_0m1779788465.3.1.min.js HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [26/May/2026 09:42:48] "GET /_dash-component-suites/dash/deps/polyfill@7.v4_1_0m1779788465.12.1.min.js HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [26/May/2026 09:42:48] "GET /_dash-component-suites/dash/deps/react-dom@18.v4_1_0m1779788465.3.1.min.js HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [26/May/2026 09:42:48] "GET /_dash-component-suites/dash/deps/prop-types@15.v4_1_0m1779788465.8.1.min.js HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [26/May/2026 09:42:48] "GET /_dash-component-suites/dash/dash-renderer/build/dash_renderer.v4_1_0m1779788465.min.js HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [26/May/2026 09:42:48] "GET /_dash-component-suites/dash/dcc/dash_core_components.v4_1_0m1779788465.js HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [26

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
Selecting previously unselected package libatk1.0-data.
(Reading database ... 118212 files and directories currently installed.)
Preparing to unpack .../00-libatk1.0-data_2.36.0-3build1_all.deb ...
Unpacking libatk1.0-data (2.36.0-3build1) ...
Selecting previously unselected package libatk1.0-0:amd64.
Preparing to unpack .../01-libatk1.0-0_2.36.0-3build1_amd64.deb ...
Unpacking libatk1.0-0:amd64 (2.36.0-3build1) ...
Selecting previously unselected package libatspi2.0-0:amd64.
Preparing to unpack .../02-libatspi2.0-0_2.44.0-3_amd64.deb ...
Unpacking libatspi2.0-0:amd64 (2.44.0-3) ...
Selecting previously unselected package libatk-bridge2.0-0:amd64.
Preparing to unpack .../03-libatk-bridge2.0-0_2.38.0-3_amd64.deb ...
Unpacking libatk-bridge2.0-0:amd64 (2.38.0-3) ...
Selecting previously unselected pack

In [ ]:
import os
import re
import time
from collections import Counter

from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC


# где может лежать бинарник Chrome на разных платформах
CHROME_PATHS = [
    '/usr/bin/google-chrome',
    '/usr/bin/google-chrome-stable',
    '/usr/bin/chromium-browser',
    '/usr/bin/chromium',
]


class SteamCardGameParser:

    # tag 1666 = Card Game на Steam (строго карточные, без AAA в которых "немного пазлов")
    BASE_URL = 'https://store.steampowered.com/search/?tags=1666&supportedlang=english&cc=us&page={page}'
    REVIEW_RE = re.compile(r'(\d+)%.+?([\d,]+) user reviews')

    def __init__(self, max_pages=3, headless=True, wait_sec=2):
        self.max_pages = max_pages
        self.headless = headless
        self.wait_sec = wait_sec
        self.driver = None

    def _start(self):
        opts = Options()
        if self.headless:
            opts.add_argument('--headless=new')
        opts.add_argument('--no-sandbox')
        opts.add_argument('--disable-dev-shm-usage')
        opts.add_argument('--window-size=1280,1800')
        # на Linux/Colab надо явно указать путь к Chrome, на винде селениум сам найдет
        for p in CHROME_PATHS:
            if os.path.exists(p):
                opts.binary_location = p
                break
        self.driver = webdriver.Chrome(options=opts)

    def _safe_text(self, parent, selector):
        # вытащить .text по селектору, если нет - вернуть None
        try:
            return parent.find_element(By.CSS_SELECTOR, selector).text.strip()
        except Exception:
            return None

    def _parse_card(self, card):
        # одна карточка игры в результатах поиска
        row = {
            'app_id': card.get_attribute('data-ds-appid'),
            'title': self._safe_text(card, '.title'),
            'release_date': self._safe_text(card, '.search_released'),
            'review_summary': None,
            'review_percent': None,
            'review_count': None,
            'price': None,
            'price_text': None,
        }
        # сводка по отзывам сидит в data-tooltip-html
        try:
            rs = card.find_element(By.CSS_SELECTOR, '.search_review_summary')
            tip = rs.get_attribute('data-tooltip-html') or ''
            parts = tip.split('<br>')
            if parts:
                row['review_summary'] = parts[0].strip() or None
            if len(parts) > 1:
                m = self.REVIEW_RE.search(parts[1])
                if m:
                    row['review_percent'] = int(m.group(1))
                    row['review_count'] = int(m.group(2).replace(',', ''))
        except Exception:
            pass
        # цена в центах в data-price-final
        try:
            pc = card.find_element(By.CSS_SELECTOR, '.search_price_discount_combined')
            cents = pc.get_attribute('data-price-final')
            if cents is not None and cents.isdigit():
                row['price'] = int(cents) / 100
            row['price_text'] = self._safe_text(pc, '.discount_final_price')
        except Exception:
            pass
        return row

    def parse(self):
        self._start()
        rows = []
        for page in range(1, self.max_pages + 1):
            url = self.BASE_URL.format(page=page)
            self.driver.get(url)
            try:
                WebDriverWait(self.driver, 10).until(
                    EC.presence_of_element_located((By.CSS_SELECTOR, 'a.search_result_row'))
                )
            except Exception:
                print(f'страница {page}: карточки не появились, прерываемся')
                break
            time.sleep(self.wait_sec)
            cards = self.driver.find_elements(By.CSS_SELECTOR, 'a.search_result_row')
            if not cards:
                print(f'страница {page}: пусто, выходим')
                break
            for c in cards:
                try:
                    rows.append(self._parse_card(c))
                except Exception:
                    continue
            print(f'страница {page}: спарсили {len(cards)} карточек')
        return pd.DataFrame(rows)

    def close(self):
        if self.driver is not None:
            try:
                self.driver.quit()
            except Exception:
                pass
            self.driver = None

In [ ]:
# запускаем парсер на 3 страницах - 25 карточек на странице, итого 75 игр
parser = SteamCardGameParser(max_pages=3)
try:
    steam = parser.parse()
finally:
    parser.close()

print()
print('всего карточных игр спарсили:', len(steam))
steam.head()

страница 1: спарсили 25 карточек
страница 2: спарсили 25 карточек
страница 3: спарсили 25 карточек

всего карточных игр спарсили: 75


,app_id,title,release_date,review_summary,review_percent,review_count,price,price_text
0,3892270,Gamble With Your Friends,"May 1, 2026",Very Positive,88,5274,7.99,$7.99
1,2868840,Slay the Spire 2,"Mar 5, 2026",Very Positive,94,53503,24.99,$24.99
2,2141910,Magic: The Gathering Arena,"May 23, 2023",Mixed,58,20673,0.00,Free
3,3265700,Vampire Crawlers: The Turbo Wildcard from Vamp...,"Apr 21, 2026",Overwhelmingly Positive,95,7402,9.99,$9.99
4,1449850,Yu-Gi-Oh! Master Duel,"Jan 18, 2022",Mostly Positive,73,48013,0.00,Free


In [ ]:
# базовая статистика
print('всего игр:', len(steam))
print('с ценой:', steam['price'].notna().sum())
print('бесплатных (price=0):', (steam['price'] == 0).sum())
print('с обзорами:', steam['review_count'].notna().sum())
print()
paid = steam[steam['price'].fillna(0) > 0]
if len(paid):
    print(f'средняя цена платных: ${paid["price"].mean():.2f}')
    print(f'медиана цены платных: ${paid["price"].median():.2f}')
print()
print('распределение по review_summary:')
for k, v in Counter(steam['review_summary'].dropna()).most_common():
    print(f'  {k:25s} -> {v}')

всего игр: 75
с ценой: 75
бесплатных (price=0): 20
с обзорами: 75

средняя цена платных: $17.38
медиана цены платных: $14.99

распределение по review_summary:
  Very Positive             -> 38
  Overwhelmingly Positive   -> 15
  Mostly Positive           -> 11
  Mixed                     -> 10
  Mostly Negative           -> 1


In [ ]:
# Plotly: распределение по типу оценки
rs_counts = Counter(steam['review_summary'].dropna())
order = ['Overwhelmingly Positive', 'Very Positive', 'Positive', 'Mostly Positive',
         'Mixed', 'Mostly Negative', 'Negative', 'Very Negative', 'Overwhelmingly Negative']
ks = [k for k in order if k in rs_counts]
vs = [rs_counts[k] for k in ks]
# плюс все остальное (вдруг новые лейблы)
for k, v in rs_counts.items():
    if k not in ks:
        ks.append(k)
        vs.append(v)
px.bar(x=ks, y=vs, title='Steam: распределение карточных игр по обзорам',
       labels={'x': 'оценка', 'y': 'игр'})

In [ ]:
# Plotly: распределение цен платных игр
paid = steam[steam['price'].fillna(0) > 0]
free_n = (steam['price'] == 0).sum()
print(f'бесплатных: {free_n}, платных: {len(paid)}')
px.histogram(paid, x='price', nbins=20, title='Steam: распределение цен платных карточных игр',
             labels={'price': 'цена $', 'count': 'игр'})

бесплатных: 20, платных: 55


In [ ]:
# Plotly: топ-10 по числу отзывов
top10 = steam.dropna(subset=['review_count']).nlargest(10, 'review_count')
px.bar(top10, x='title', y='review_count',
       hover_data=['review_summary', 'review_percent', 'price'],
       title='Steam: топ-10 карточных игр по числу обзоров',
       labels={'title': 'игра', 'review_count': 'число отзывов'})

### Что видно из Steam-данных

- Спарсили ~75 карточных игр (Steam tag 1666, 3 страницы по 25 карточек)
- В выдаче: Slay the Spire, Balatro, Magic: The Gathering Arena, Yu-Gi-Oh Master Duel, Inscryption, Marvel Snap, Tabletop Simulator, всякие пасьянсы и deckbuilders
- Большая часть с оценкой Very/Overwhelmingly Positive - карточные игры на Steam обычно нишевые но обожаемые жанр
- Цены: много F2P (Magic Arena, Marvel Snap, Yu-Gi-Oh), платные обычно $5-25, есть дорогие премиум

**Отличия от Amazon-данных:**
- Amazon: физические jigsaw, US-рынок, 2001-2014, бренды-производители (Melissa & Doug, Ravensburger)
- Steam: digital карточные игры, мировой рынок, актуальные годы, бренды-разработчики/издатели
- Пересечения брендов почти нет - это два разных рынка
- Категории не совпадают (Amazon: Jigsaw/Floor/Pegged; Steam: deckbuilders/TCG/пасьянсы), но обе про "игры для мозга"

**Зачем нам это:** показать что индустрия "thinking games" живет в обоих сегментах - физический jigsaw и digital cards - и потенциальная аудитория пересекается (люди любящие головоломки и стратегические настолки).

Используем уже построенные UserAnalytics. Один юзер - одна точка, наблюдения независимые. AOV из .aov, avg_pieces из .complexity_preference['avg_pieces']. Юзеры без pieces дропаем.

Demand-side гипотеза: юзеры предпочитающие сложные пазлы имеют больший средний чек (потому что сложные пазлы дороже).

In [ ]:
from scipy.stats import linregress

In [ ]:
# собираем per-user dataframe прямо из объектов UserAnalytics
rows = []
for a in analytics_list:
    cp = a.complexity_preference
    if cp is not None:
        rows.append({'aov': a.aov, 'avg_pieces': cp['avg_pieces']})
u2 = pd.DataFrame(rows)

res2 = linregress(u2['avg_pieces'], u2['aov'])
print('А7.2 user-level: AOV ~ avg_pieces')
print(f'  AOV = {res2.slope:.5f} * avg_pieces + {res2.intercept:.3f}')
print(f'  R² = {res2.rvalue**2:.4f}, p-value = {res2.pvalue:.3e}, n = {len(u2)}')
print(f'  смысл: +100 деталей к среднему пазлу юзера -> +${res2.slope*100:.2f} к AOV')

А7.2 user-level: AOV ~ avg_pieces
  AOV = 0.00549 * avg_pieces + 12.754
  R² = 0.2295, p-value = 1.737e-87, n = 1510
  смысл: +100 деталей к среднему пазлу юзера -> +$0.55 к AOV


In [ ]:
# scatter AOV ~ avg_pieces с OLS
px.scatter(u2, x='avg_pieces', y='aov', trendline='ols', opacity=0.4,
           title=f'А7.2: AOV от avg_pieces (R²={res2.rvalue**2:.3f}, n={len(u2)})',
           labels={'avg_pieces': 'средний размер пазла у юзера', 'aov': 'AOV $'},
           trendline_color_override='red')

Используем наши классы UserAnalytics. Каждые +100 деталей в типичном пазле клиента дают +$0.55 к AOV. Сегмент любителей сложных пазлов - дороже в чеке.

In [47]:
# Сравниваем "одобрение аудитории" Amazon-пазлы vs Steam-карточные.
# Метрики приводим к одной шкале (% положительных отзывов).
from scipy.stats import ttest_ind

# Amazon: % отзывов с оценкой 4-5 звезд по каждому товару
amzn = df.dropna(subset=['overall']).copy()
amzn['recommended'] = (amzn['overall'] >= 4).astype(int)
amzn_g = amzn.groupby('asin').agg(
    satisfaction=('recommended', lambda x: x.mean() * 100),
    n=('reviewerID', 'size'),
).reset_index()

# Steam: review_percent уже в %
stm_p = steam.dropna(subset=['review_percent']).copy()
stm_p['satisfaction'] = stm_p['review_percent']


# красивый текстовый отчет
print('=' * 55)
print('  ОДОБРЕНИЕ АУДИТОРИИ: AMAZON vs STEAM')
print('=' * 55)
print()
print(f'{"метрика":<15} {"Amazon (пазлы)":>20} {"Steam (карты)":>15}')
print('-' * 55)
print(f'{"n":<15} {len(amzn_g):>20} {len(stm_p):>15}')
print(f'{"mean, %":<15} {amzn_g["satisfaction"].mean():>20.1f} {stm_p["satisfaction"].mean():>15.1f}')
print(f'{"median, %":<15} {amzn_g["satisfaction"].median():>20.1f} {stm_p["satisfaction"].median():>15.1f}')
print(f'{"std, %":<15} {amzn_g["satisfaction"].std():>20.1f} {stm_p["satisfaction"].std():>15.1f}')
print(f'{"min, %":<15} {amzn_g["satisfaction"].min():>20.1f} {stm_p["satisfaction"].min():>15.1f}')
print(f'{"max, %":<15} {amzn_g["satisfaction"].max():>20.1f} {stm_p["satisfaction"].max():>15.1f}')
print()
print(f'разница средних: 2.9 п.п. (Amazon - Steam)')


  ОДОБРЕНИЕ АУДИТОРИИ: AMAZON vs STEAM

метрика               Amazon (пазлы)   Steam (карты)
-------------------------------------------------------
n                                373              75
mean, %                         87.3            84.3
median, %                       92.3            87.0
std, %                          15.1            11.7
min, %                          33.3            37.0
max, %                         100.0            98.0

разница средних: 2.9 п.п. (Amazon - Steam)
